In [ ]:
%load_ext autoreload
%autoreload 2

# Qualitative Evaluation MicroSAM

Evaluates automatic and prompt based segmentation using micro-sam

## Set-up Notebook

### Check GPU

In [ ]:
import torch

def check_pytorch_gpu():
    """
    Checks if a PyTorch-compatible GPU is installed and available.
    """
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        print(f"GPU is available. Found {gpu_count} GPU(s).")
        for i in range(gpu_count):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        return True
    else:
        # A check for Apple Silicon (MPS)
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
             print("MPS device (Apple Silicon GPU) is available.")
             return True
        print("No PyTorch-compatible GPU found. PyTorch will use the CPU.")
        return False
    
check_pytorch_gpu()

In [ ]:
from micro_sam.util import get_device

get_device()

### Download a microsam model

Here we get the Model for cells and nuclei in light microscopy data with ViT Large image encoder.

In [ ]:
from pathlib import Path
from micro_sam.util import get_cache_directory, get_device, get_sam_model
import os
import torch

# Change where the checkpoints are saved
os.environ['MICROSAM_CACHEDIR'] = '~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam'
print(get_cache_directory())

# Set model and device choice
sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
decoder_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
model_type = 'vit_b_lm'
device = get_device()
print(device)

# Download model
sam = get_sam_model(
    model_type = model_type, # specify the model type
    device = device, # automatically loads the model to device
)
print(type(sam))
print(sam.device)

### Define helper functions

In [ ]:
from micro_sam.automatic_segmentation import automatic_instance_segmentation
from micro_sam import util
import torch
from micro_sam.instance_segmentation import AMGBase, InstanceSegmentationWithDecoder, get_decoder, get_amg
from typing import Optional, Union, Tuple
import os
import numpy as np

def get_predictor_and_segmenter(
    model_type: str,
    checkpoint_path: Union[os.PathLike, str],
    decoder_path: Union[os.PathLike, str],
    device: str = None,
    amg: Optional[bool] = None,
    is_tiled: bool = False,
    **kwargs,
) -> Tuple[util.SamPredictor, Union[AMGBase, InstanceSegmentationWithDecoder]]:
    """Get the Segment Anything model and class for automatic instance segmentation.

    Args:
        model_type: The Segment Anything model choice.
        checkpoint: The filepath to the stored model checkpoints.
        device: The torch device. By default, automatically chooses the best available device.
        amg: Whether to perform automatic segmentation in AMG mode.
            Otherwise AIS will be used, which requires a special segmentation decoder.
            If not specified AIS will be used if it is available and otherwise AMG will be used.
        is_tiled: Whether to return segmenter for performing segmentation in tiling window style.
            By default, set to 'False'.
        kwargs: Keyword arguments for the automatic mask generation class.

    Returns:
        The Segment Anything model.
        The automatic instance segmentation class.
    """
    # Get the device
    if device is None:
        device = util.get_device(device=device)

    # Get the predictor and state for Segment Anything models.
    predictor, state = util.get_sam_model(
        model_type=model_type, device=device, checkpoint_path=checkpoint_path, return_state=True,
    )
    state["decoder_state"] = torch.load(decoder_path, map_location=device, weights_only=False)

    if amg is None:
        amg = "decoder_state" not in state

    if amg:
        decoder = None
    else:
        if "decoder_state" not in state:
            raise RuntimeError("You have passed 'amg=False', but your model does not contain a segmentation decoder.")
        decoder_state = state["decoder_state"]
        decoder = get_decoder(image_encoder=predictor.model.image_encoder, decoder_state=decoder_state, device=device)

    segmenter = get_amg(predictor=predictor, is_tiled=is_tiled, decoder=decoder, **kwargs)

    return predictor, segmenter

def run_automatic_instance_segmentation(
    image: np.ndarray,
    ndim: int,
    checkpoint_path: Union[os.PathLike, str],
    decoder_path: Union[os.PathLike, str],
    model_type: str = "vit_b_lm",
    device: Optional[Union[str, torch.device]] = None,
    tile_shape: Optional[Tuple[int, int]] = None,
    halo: Optional[Tuple[int, int]] = None,
):
    """Automatic Instance Segmentation (AIS) by training an additional instance decoder in SAM.

    NOTE: AIS is supported only for `µsam` models.

    Args:
        image: The input image.
        ndim: The number of dimensions for the input data.
        checkpoint_path: The path to stored checkpoints.
        model_type: The choice of the `µsam` model.
        device: The device to run the model inference.
        tile_shape: The tile shape for tiling-based segmentation.
        halo: The overlap shape on each side per tile for stitching the segmented tiles.

    Returns:
        The instance segmentation.
    """
    # Step 1: Get the 'predictor' and 'segmenter' to perform automatic instance segmentation.
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=model_type,  # choice of the Segment Anything model
        checkpoint_path=checkpoint_path,  # overwrite to pass your own finetuned model.
        decoder_path=decoder_path,
        device=device,  # the device to run the model inference.
        amg=False,  # set the automatic segmentation mode to AIS.
        is_tiled=(tile_shape is not None),  # whether to run automatic segmentation with tiling.
    )

    # Step 2: Get the instance segmentation for the given image.
    prediction = automatic_instance_segmentation(
        predictor=predictor,  # the predictor for the Segment Anything model.
        segmenter=segmenter,  # the segmenter class responsible for generating predictions.
        input_path=image,  # the filepath to image or the input array for automatic segmentation.
        ndim=ndim,  # the number of input dimensions.
        tile_shape=tile_shape,  # the tile shape for tiling-based prediction.
        halo=halo,  # the overlap shape for tiling-based prediction.
    )

    return prediction


def run_automatic_mask_generation(
    image: np.ndarray,
    ndim: int,
    checkpoint_path: Union[os.PathLike, str],
    decoder_path: Union[os.PathLike, str],
    model_type: str = "vit_b",
    device: Optional[Union[str, torch.device]] = None,
    tile_shape: Optional[Tuple[int, int]] = None,
    halo: Optional[Tuple[int, int]] = None,
):
    """Automatic Mask Generation (AMG) is the automatic segmentation method offered by SAM.

    NOTE: AMG is supported for both Segment Anything models and `µsam` models.

    Args:
        image: The input image.
        ndim: The number of dimensions for the input data.
        checkpoint_path: The path to stored checkpoints.
        model_type: The choice of the SAM / `µsam` model.
        device: The device to run the model inference.
        tile_shape: The tile shape for tiling-based segmentation.
        halo: The overlap shape on each side per tile for stitching the segmented tiles.

    Returns:
        The instance segmentation.
    """
    # Step 1: Get the 'predictor' and 'segmenter' to perform automatic instance segmentation.
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=model_type,  # choice of the Segment Anything model
        checkpoint_path=checkpoint_path,  # overwrite to pass your own finetuned model.
        decoder_path = decoder_path,
        device=device,  # the device to run the model inference.
        amg=True,  # set the automatic segmentation mode to AMG.
        is_tiled=(tile_shape is not None),  # whether to run automatic segmentation with tiling.
    )

    # Step 2: Get the instance segmentation for the given image.
    prediction = automatic_instance_segmentation(
        predictor=predictor,  # the predictor for the Segment Anything model.
        segmenter=segmenter,  # the segmenter class responsible for generating predictions.
        input_path=image,  # the filepath to image or the input array for automatic segmentation.
        ndim=ndim,  # the number of input dimensions.
        tile_shape=tile_shape,  # the tile shape for tiling-based prediction.
        halo=halo,  # the overlap shape for tiling-based prediction.
    )

    return prediction

## Automatic Instance Segmentation (AIS)

Runs AIS for the tif images inside the folder. Should be max, sharpest, or subtract projections.

### Load file list

In [ ]:
from load_files import find_files_by_pattern

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SUBTRACT*.tif"
    )

max_tif_files = find_files_by_pattern(search_path[0], file_pattern[2], verbose=True)

### Compute AIS

In [ ]:
from preprocessing import preprocess_image
from pathlib import Path
from micro_sam.util import get_device
from visualize import save_segmentation_visualization_AIS

# Model directories
sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
decoder_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
model_type = 'vit_l_lm'
device = get_device()

# Save outputs
all_processed_images = {}
all_generated_masks = {}

# Compute Loop
for file_path in max_tif_files:
    file_name = file_path.name
    print(f"Processing {file_name}...")
    
    processed_image, _, _= preprocess_image(
        file_path,
        scale_intensity=True,
        resize_image=True,
        max_dim=1024,
        convert_to_rgb=True,
        prompt_mode='none'
    )
    
    if processed_image is None:
        print(f"  Skipping {file_name} due to preprocessing error.")
        continue
    
    all_processed_images[file_name] = processed_image

    prediction = run_automatic_instance_segmentation(
        image = processed_image,
        ndim = 2,
        checkpoint_path = str(sam_checkpoint),
        decoder_path = str(decoder_checkpoint),
        model_type = model_type,
        device = device,
        tile_shape = None,
        halo = None,
    )
    
    if prediction is not None:
        all_generated_masks[file_name] = prediction
        print(f"  Generated automatic instance segmentation for {file_name}.")
        
# Visualization loop
# Define a base input path to calculate relative paths from.
# This should be the common parent directory of your search paths.
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/microSAM_AIS")
print(f"\nComparison visualizations will be saved to: {base_output_dir.resolve()}")

for file_path in max_tif_files:
    file_name = file_path.name

    if file_name not in all_generated_masks:
        continue
    
    # 1. Determine the relative path of the input file's folder.
    try:
        relative_dir = file_path.parent.relative_to(base_input_dir)
    except ValueError:
        # Fallback if the file is not in the base_input_dir
        relative_dir = file_path.parent.name

    # 2. Create the full output directory and the final filename.
    current_output_dir = base_output_dir / relative_dir
    current_output_dir.mkdir(parents=True, exist_ok=True)
    output_filename = current_output_dir / f"{file_path.stem}_visualization.png"
    
    print(f"Creating visualization for {file_name}...")
    
    save_segmentation_visualization_AIS(
        processed_image=all_processed_images[file_name],
        masks=all_generated_masks[file_name],
        output_path=output_filename,
        title=f"microSAM AIS Output for {file_name}"
    )
    
    print(f"  -> Saved to {output_filename}")

print("\nAll comparison visualizations have been saved.")

## Compare Slice Subtraction Methods

Compares two slice subtraction methods. The goal is to enhance contrast in BF image by subtracting the slices around the in-focus slice. The `direct` method subtracts the slice below the focal plane from the slice above. The `average` method uses the formula $\text{In Focus Slice}-\frac{1}{2}(\text{Slice Above}+\text{Slice Below})$. Additionally, an offset parameter sets how many slices to go above and below the focal plane.

The comparison is only done for `CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF.tif` channels. There are 30 images: 5 channels * 2 subtraction methods * 3 offsets (1,5,10)

### Load file list

In [ ]:
from load_files import find_files_by_pattern

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    )

file_pattern = (
    "*SUBTRACT*.tif",
    )

max_tif_files = find_files_by_pattern(search_path[0], file_pattern[0], verbose=True)

### Run AIS Comparison

In [ ]:
from preprocessing import preprocess_image
from pathlib import Path
from micro_sam.util import get_device

# Model directories
sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
decoder_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
model_type = 'vit_l_lm'
device = get_device()

# Save outputs
all_processed_images = {}
all_generated_masks = {}

# Compute Loop
for file_path in max_tif_files:
    file_name = file_path.name
    print(f"Processing {file_name}...")
    
    processed_image, _, _= preprocess_image(
        file_path,
        scale_intensity=True,
        resize_image=True,
        max_dim=1024,
        convert_to_rgb=True,
        generate_prompt_mask=False
    )
    
    if processed_image is None:
        print(f"  Skipping {file_name} due to preprocessing error.")
        continue
    
    all_processed_images[file_name] = processed_image

    prediction = run_automatic_instance_segmentation(
        image = processed_image,
        ndim = 2,
        checkpoint_path = str(sam_checkpoint),
        decoder_path = str(decoder_checkpoint),
        model_type = model_type,
        device = device,
        tile_shape = None,
        halo = None,
    )
    
    if prediction is not None:
        all_generated_masks[file_name] = prediction
        print(f"  Generated automatic instance segmentation for {file_name}.")

### Run visualization loop

In [ ]:
import re
from visualize import save_subtraction_comparison_visualization
from collections import defaultdict

# Visualization loop
# --- Configuration for Subtraction Comparison ---
SUBTRACTION_METHODS = ['direct', 'average']
OFFSETS = [1, 5, 10]
SHARPNESS_METHOD = 'normalized_variance'

# This should be the common parent directory for all your search_path entries.
base_input_dir = Path("~/A8/Data_Roan/").expanduser()

base_comparison_output_dir = Path("output/microSAM_AIS_Subtraction_Comparison")
base_comparison_output_dir.mkdir(parents=True, exist_ok=True)
print(f"\nSubtraction comparison visualizations will be saved to: {base_comparison_output_dir.resolve()}")

# --- New Logic: Group files by experiment and channel before plotting ---
grouped_files = defaultdict(list)
for file_path in max_tif_files:
    # Extract the core experiment name and channel to group files
    match = re.search(r'SUBTRACT-.*?_C(\d+)_(.*)\.tif', file_path.name)
    if match:
        channel_num = match.group(1)
        base_name = match.group(2)
        key = (base_name, channel_num, file_path.parent)
        grouped_files[key].append(file_path)
    else:
        print(f"Warning: Could not parse base name and channel from {file_path.name}. Skipping.")

for (base_name, channel_num, parent_dir), file_paths in grouped_files.items():
    print(f"\nProcessing group: {base_name} - Channel C{channel_num}")

    # --- This dictionary will hold the images for the plot ---
    subtracted_images_for_channel = {}
    ais_masks_for_channel = {}

    for file_path in file_paths:
        file_name = file_path.name
        if file_name not in all_processed_images:
            print(f"  Skipping {file_name}: Preprocessed image not found in dictionary.")
            continue

        # Use regex to determine the method and offset from the filename
        match = re.search(r'SUBTRACT-(direct|average)-.*?O(\d+)_', file_name)
        if match:
            method = match.group(1)
            offset = int(match.group(2))
            key = (method, offset)
            
            subtracted_images_for_channel[key] = all_processed_images[file_name]
            if file_name in all_generated_masks:
                ais_masks_for_channel[key] = all_generated_masks[file_name]
            print(f"  Found pre-processed image for: {method}, offset {offset}")
        else:
            print(f"  Warning: Could not parse method/offset from {file_name}")

    if not subtracted_images_for_channel or not ais_masks_for_channel:
        print("  Skipping plot generation: Missing necessary images or masks for this group.")
        continue

    # Determine the relative path for saving the output, using the first file in the group as a reference
    try:
        relative_dir = parent_dir.relative_to(base_input_dir)
    except ValueError:
        relative_dir = parent_dir.name

    current_comparison_output_dir = base_comparison_output_dir / relative_dir
    current_comparison_output_dir.mkdir(parents=True, exist_ok=True)
    
    output_filename = current_comparison_output_dir / f"C{channel_num}_{base_name}_subtraction_comparison.png"
    
    print(f"Creating subtraction comparison visualization...")
    
    save_subtraction_comparison_visualization(
        ais_masks_dict=ais_masks_for_channel,
        subtracted_images_dict=subtracted_images_for_channel,
        output_path=output_filename,
        title=f"microSAM AIS Subtraction Comparison for {base_name} - Channel C{channel_num}",
        offsets=OFFSETS
    )
    print(f"  -> Saved comparison plot to {output_filename}")

print("\nAll comparison visualizations have been saved.")

In [ ]:
for i in grouped_files.keys():
    print(i)

## Prompt based segmentation

Outputs a binary mask. No automatic instance segmentation.

### Load file list

In [ ]:
from load_files import find_files_by_pattern

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SUBTRACT*.tif"
    )

max_tif_files = find_files_by_pattern(search_path[2], file_pattern[1], verbose=True)

### Run prompt based segmentation

In [ ]:
from micro_sam.inference import batched_inference
from micro_sam.util import get_sam_model, get_device
import numpy as np
from pathlib import Path
import os
from preprocessing import preprocess_image

# --- Setup ---
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
model_type = 'vit_l_lm'
device = get_device()

# Load model
sam = get_sam_model(
    model_type = model_type, # specify the model type
    device = device, # automatically loads the model to device
    checkpoint_path = sam_checkpoint
)

# print(type(sam))
print(sam.device)

# --- Main Processing Loop ---
all_processed_images = {}
all_prompts = {}
all_prompt_types = {}
all_generated_masks = {}
all_generated_scores = {}
all_generated_logits = {}

# --- CHOOSE YOUR PROMPT MODE ---
PROMPT_MODE = 'points' # Can be 'points', 'mask', or 'bbox'

# --- Find DAPI Path ---
# Dynamically find the index for the DAPI file (e.g., C4-Composite.tif)
try:
    DAPI_FILE_INDEX = [i for i, f in enumerate(max_tif_files) if 'C5' in f.name][0]
except IndexError:
    raise FileNotFoundError("DAPI file with 'C4' in its name not found in the channel files.")

dapi_path = None
if PROMPT_MODE in ('points', 'bbox') and DAPI_FILE_INDEX is not None:
    # Fetches the corresponding DAPI file path
    dapi_path = max_tif_files[DAPI_FILE_INDEX]
    print(f"DAPI file path is {str(dapi_path)}")

print(f"Starting mask generation for {len(max_tif_files)} files using '{PROMPT_MODE}' prompts...")

for file_path in max_tif_files:
    file_name = file_path.name
    print(f"Processing {file_name}...")

    # 1. Preprocess the image and generate prompts
    processed_image, prompts, prompt_type = preprocess_image(
        file_path,
        scale_intensity=True,
        resize_image=True,
        max_dim=1024,
        convert_to_rgb=True,
        prompt_mode=PROMPT_MODE,
        dapi_image_path=dapi_path,
        min_mask_area=150,
        use_watershed=True,
        bbox_radius_multiplier=2
    )

    if processed_image is None or prompts is None:
        print(f"  Skipping {file_name} due to preprocessing/prompting error.")
        continue

    # Store results for visualization
    all_processed_images[file_name] = processed_image
    all_prompt_types[file_name] = prompt_type
    all_prompts[file_name] = prompts

    # 2. Set image for the predictor
    sam.set_image(processed_image)
    
    # 3. Predict masks based on the prompt type
    masks, scores, logits = (None, None, None) # Initialize
    
    if prompt_type == 'points':
        # batched_inference expects points in (N, 1, 2) shape and (x, y) order.
        # Our preprocessor now provides points in the correct (x, y) order.
        # We just need to reshape from (N, 2) to (N, 1, 2).
        points_for_batch = prompts[:, None, :]
        
        mask_data = batched_inference(
            predictor=sam,
            image=processed_image,
            batch_size=len(prompts),
            points=points_for_batch,
            point_labels=np.ones((len(prompts), 1)),
            multimasking=False, # We want one mask per point
            return_instance_segmentation=False # Return list of dicts
        )
        
        masks = [d['segmentation'].cpu().numpy() for d in mask_data]
        scores = [d['predicted_iou'] for d in mask_data]
        logits = [np.squeeze(d['logits'].cpu().numpy()) for d in mask_data]
    elif prompt_type == 'bbox':
        # Use batched_inference for multiple bounding boxes.
        # It returns a list of dictionaries, one for each prompt.
        mask_data = batched_inference(
            predictor=sam,
            image=processed_image,
            batch_size=len(prompts), # Process all boxes in one go
            boxes=prompts,
            multimasking=False, # We want one mask per box
            return_instance_segmentation=False # Return list of dicts
        )
        
        # Extract the masks, scores, and logits from the returned data
        masks = [d['segmentation'].cpu().numpy() for d in mask_data]
        scores = [d['predicted_iou'] for d in mask_data]
        logits = [np.squeeze(d['logits'].cpu().numpy()) for d in mask_data]
        
    # Add logic for 'mask' prompt if needed
    
    # 4. Save the outputs for visualization
    if masks is not None:
        # 'masks' is a list of individual mask arrays (one for each prompt).
        all_generated_masks[file_name] = masks
        # 'scores' is a list of scores.
        all_generated_scores[file_name] = scores
        # 'logits' is a list of logit maps.
        all_generated_logits[file_name] = logits
        print(f"  Generated {len(all_generated_masks[file_name])} masks for {file_name}.")
    else:
        print(f"  Mask generation failed for {file_name}.")

print("\nFinished generating masks for all files.")

In [ ]:
from visualize import save_multi_mask_visualization

# --- Visualization Loop ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path(f"output/microSAM_{PROMPT_MODE}_prompts")
print(f"\\nComparison visualizations will be saved to: {base_output_dir.resolve()}")

for file_path in max_tif_files:
    file_name = file_path.name

    if file_name not in all_generated_masks:
        continue

    # 1. Determine the relative path of the input file's folder.
    try:
        relative_dir = file_path.parent.relative_to(base_input_dir)
    except ValueError:
        relative_dir = file_path.parent.name

    # 2. Create the full output directory and the final filename.
    current_output_dir = base_output_dir / relative_dir
    current_output_dir.mkdir(parents=True, exist_ok=True)
    output_filename = current_output_dir / f"{file_path.stem}_visualization.png"
    
    print(f"Creating visualization for {file_name}...")
    
    save_multi_mask_visualization(
        processed_image=all_processed_images[file_name],
        prompts=all_prompts[file_name],
        prompt_type=all_prompt_types[file_name],
        masks=all_generated_masks[file_name],
        scores=all_generated_scores[file_name],
        logits=all_generated_logits[file_name],
        output_path=output_filename,
        title=f"SAM Output for {file_name}"
    )
    
    print(f"  -> Saved to {output_filename}")

print("\nAll comparison visualizations have been saved.")

In [ ]:
from visualize import save_channel_comparison_visualization

# --- Visualization Loop ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path(f"output/microSAM_{PROMPT_MODE}_prompts_comparison")
print(f"\nComparison visualizations will be saved to: {base_output_dir.resolve()}")

# Since we are creating one visualization per FOV (Field of View), we only need to run the loop once.
# We assume all files in `max_tif_files` belong to the same FOV.
if max_tif_files:
    # 1. Determine the relative path of the input file's folder.
    # We use the first file to determine the directory structure.
    first_file_path = max_tif_files[0]
    try:
        relative_dir = first_file_path.parent.relative_to(base_input_dir)
    except ValueError:
        relative_dir = first_file_path.parent.name

    # 2. Create the full output directory and the final filename for the combined plot.
    current_output_dir = base_output_dir / relative_dir
    current_output_dir.mkdir(parents=True, exist_ok=True)
    # The output filename can be based on the directory name (e.g., FOV1)
    output_filename = current_output_dir / f"{first_file_path.parent.name}_channel_comparison.png"
    
    print(f"Creating combined channel visualization for {relative_dir}...")
    
    # Call the new visualization function that handles all channels at once
    save_channel_comparison_visualization(
        all_processed_images=all_processed_images,
        all_prompts=all_prompts,
        all_prompt_types=all_prompt_types,
        all_generated_masks=all_generated_masks,
        all_generated_scores=all_generated_scores,
        all_generated_logits=all_generated_logits,
        file_paths=max_tif_files,
        output_path=output_filename,
        title=f"Micro-SAM Channel Comparison for {relative_dir}"
    )
    
    print(f"  -> Saved to {output_filename}")

## Testing class implementation

### Prompt based segmentation

In [ ]:
# Imports for logging and the pipeline
import logging
import sys
from pathlib import Path
from micro_sam_pipeline import MicroSAMPipeline, load_config

# 1. Get the root logger.
root_logger = logging.getLogger()

# 2. If you have run cells before, this clears any existing handlers to prevent duplicates.
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)

# 3. Configure the root logger. This will capture logs from all modules (micro_sam_pipeline, prompt_generation, etc.).
#    - level: The minimum level of message to display. Use logging.INFO for standard output,
#             or logging.DEBUG for more detailed, verbose output.
#    - format: The layout of each log message.
#    - stream: Where to send the output (sys.stdout is the notebook's output).
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)
# You can change the level to logging.DEBUG for more verbose output
# logging.getLogger().setLevel(logging.DEBUG)

# --- User Configuration ---
# Please edit the paths and indices in this section before running.

# 3. Load the main configuration from your JSON file
# Make sure your micro_sam_config.json file is correctly set up with model paths.
try:
    config = load_config("micro_sam_config.json")
except FileNotFoundError as e:
    logging.error(f"Configuration file not found. Please ensure 'micro_sam_config.json' is in the correct directory.")
    # Stop execution if config is missing
    raise e

# 4. Set the pipeline to run prompt-based segmentation with bounding boxes
config["segmentation_mode"] = "prompted"
config["prompting"]["prompt_mode"] = "points" # bbox or points, mask in implementation process
config["prompting"]["bbox_radius_multiplier"] = 3.5

config['preprocessing']['resize_image'] = False

config['base_input_dir'] = Path("~/A8/Data_Roan/").expanduser()
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
config['model_type'] = 'vit_l_lm'

# 5. Initialize the pipeline by pointing it to your data directory.
# This will find all matching files and print a numbered list for your reference.
search_paths = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_01_CY5, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_DIC",
    )

file_patterns = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SUBTRACT-direct*O5*.tif"
    )

# TODO: Replace with the actual path to your image data folder.
search_path = search_paths[6]
# TODO: Replace with the file pattern you want to find.
file_pattern = file_patterns[0]

pipeline = MicroSAMPipeline(search_path, file_pattern, config)

In [ ]:
# 6. Define your analysis plan using the indices printed above.
# microsam takes 3 channels
analysis_plan = [
    [0,0,2],
    [0,2,2],
    [0,1,2]
]

# 7. Execute the segmentation runs defined in your analysis plan.
# The pipeline will use the DAPI channel at the specified index to generate prompts
# and will cache image data to avoid reloading files used in multiple runs.
logging.info("Starting segmentation runs...")
pipeline.run(run_spec=analysis_plan)
logging.info("All segmentation runs completed.")

# 8. Visualize the results.
# This will save a separate visualization for each run defined in the analysis_plan.
# Composite runs will be saved to a folder ending in '_comp'.
logging.info("Generating visualizations...")
pipeline.visualize_results(visualization_mode='channel_comparison') # or single
logging.info("Visualizations have been saved to the 'output' directory.")

In [ ]:
import cv2
from image_processing_tools.image_class.image_container import ImageContainer
from image_processing_tools.util.load_files import find_files_by_pattern
from skimage.filters import threshold_li
import matplotlib.pyplot as plt

files = find_files_by_pattern("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI", '*MAX*.tif', verbose=True)

config['preprocessing']['quantization'] = '8bit'

img = ImageContainer(files[2],config).merge()
thresh_val = threshold_li(img)
_, binary_mask = cv2.threshold(img, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.imshow(binary_mask)